# Bandcamp Artist Discog Scraper
Based on:
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)
- diprog's [tls-client-async](https://github.com/diprog/python-tls-client-async) for better client identifiers.
- greasyfork's [Bandcamp: Show more dates](https://greasyfork.org/en/scripts/537005-bandcamp-show-more-dates/code)
- Shoutout to: dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py)

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

In [11]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import logging
from rich.logging import RichHandler

FORMAT = '%(message)s'
logging.basicConfig(
    level="INFO", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
)
log = logging.getLogger('rich')

BASE_DIR = Path.cwd()
LINKS_FILE = BASE_DIR / 'slushwave-bandcamp-links.txt'
OUTPUT_FILE = BASE_DIR / 'bc-albums-image-links.csv'
IMAGES_DIR = BASE_DIR / 'images'

### Use `tls-client` instead of plain requests
TLS fingerprinting gives a more browser-like behavior to bypass anti-bot protections.

In [12]:
from firefox_profiles import FINGERPRINTS
from bs4 import BeautifulSoup
from async_tls_client import AsyncSession
import asyncio
import os
import random
import json

In [13]:
# #STARTUP TEST PROFILES
# TEST_URL = 'https://giftsfromhome.bandcamp.com/album/-'
# async def test_profile(profile):
# 	s = AsyncSession(client_identifier=profile)
# 	r = await s.get(TEST_URL)
# 	soup = BeautifulSoup(r.text, 'lxml') # type: ignore
# 	return soup.title and soup.title.get_text(strip=True) == "Client Challenge"

# tasks = [test_profile(profile) for profile in FINGERPRINTS]
# results = await asyncio.gather(*tasks, return_exceptions=True)

# failed = []
# OK_CLIENTS = []
# for profile, result in zip(FINGERPRINTS, results):
# 	if result is True:
# 		failed.append(profile)
# 		print(profile, "ERROR", result)
# 	else:
# 		OK_CLIENTS.append(profile)

# print("Good Profiles:")
# print(OK_CLIENTS)

In [14]:
GOOD_PROFILES = Path("good_profiles.json")
with open(GOOD_PROFILES, "r") as f:
		OK_CLIENTS = json.load(f)
class BrowserSession:
	def __init__(self):
		self.requests_made = 0
		self.new_session()
		self.retire_after = random.randint(40, 100)

	def rotate_client(self):
		self.client_identifier = random.choice(OK_CLIENTS)

	def new_session(self):
		self.rotate_client()

		self.session = AsyncSession(
			client_identifier=self.client_identifier,
			random_tls_extension_order=True
		)

		self.session.proxies.update({
			"http": os.getenv("mobileproxyuk"),
			"https": os.getenv("mobileproxyuk"),
		})

		self.session.headers.update({
			"Referer": "https://bandcamp.com/",
			"Accept-Language": "en-US,en;q=0.9",
		})

	async def get(self, url, **kwargs):
		if self.requests_made >= self.retire_after:
			self.new_session()
			self.requests_made = 0
			self.retire_after = random.randint(40, 100)

		resp = await self.session.get(url,**kwargs)
		self.requests_made += 1
		return resp

In [15]:
import re
def nozero(text):
	return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)

In [16]:
urls = [
	'https://desertsand.bandcamp.com/album/perli-tal-passat',
	'https://noproblematapes.bandcamp.com/album/--89',
	# 'https://geometriclullaby.bandcamp.com/album/geo-c07'
	]

random.seed(42)
s = BrowserSession()
failed = []
soups = []

for url in urls:
	r = await s.get(url)
	soup = BeautifulSoup(r.text, 'lxml')
	if soup.title and soup.title.get_text(strip=True) == "Client Challenge":
		failed.append({
			"url": url,
			"profile": s.client_identifier,
			})
		log.warning(f"Client Challenge with {s.client_identifier} for {url}")
		s.new_session()
		continue
	soups.append(soup)

print("Albums fetched:")
print([soup.title.get_text(strip=True) for soup in soups])

# tasks = [s.get(url) for url in urls]
# await asyncio.gather(*tasks)

Albums fetched:
['Perli tal-Passat | desert sand feels warm at night & days of blue skies | desert sand feels warm at night', 'ලෝක දෙකක් අතර | days of blue skies | NO PROBLEMA TAPES']


In [ ]:
soup = soups[1]
schema = json.loads(soup.select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soup.select_one("[data-tralbum]")
schema.keys()

dict_keys(['albumReleaseType', '@id', 'mainEntityOfPage', '@type', 'name', 'dateModified', 'albumRelease', 'byArtist', 'publisher', 'numTracks', 'track', 'image', 'keywords', 'datePublished', 'description', 'creditText', 'copyrightNotice', 'comment', 'sponsor', 'additionalProperty', '@context'])

In [18]:
schema['byArtist']['name']

'days of blue skies'

### Metadata of one track

In [19]:
t_url = 'https://noproblematapes.bandcamp.com/track/feat-memorykeeper7'
r = await s.get(t_url)
soup = BeautifulSoup(r.text, 'lxml')
t_schema = json.loads(soup.select("script[type='application/ld+json']")[0].get_text(strip=True))
t_schema.keys()

dict_keys(['@type', '@id', 'additionalProperty', 'name', 'duration', 'dateModified', 'datePublished', 'inAlbum', 'byArtist', 'publisher', 'copyrightNotice', 'keywords', 'image', 'mainEntityOfPage', '@context'])

In [20]:
[
  t_schema['byArtist']['name'],
	t_schema['name'],
  t_schema['image']
]

['NO PROBLEMA TAPES',
 'لا طريق آخر (feat. memorykeeper7)',
 'https://f4.bcbits.com/img/a2569838638_10.jpg']

In [21]:
tralbum_tag = soup.select_one("[data-tralbum]")
t_tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}"))
t_tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'album_url', 'album_upsell_url', 'url'])

In [22]:
t_tralbum.get('art_id')

2569838638

In [23]:
	# async def get_track_data(self, url):
	# 	async with self.sem:
	# 		r = await self.s.get(url)
	# 		soup = BeautifulSoup(r.text or "", "lxml")

	# 		try:
	# 			t_schema = json.loads(soup.select_one("script[type='application/ld+json']").get_text(strip=True))
	# 		except Exception:
	# 			log.exception(f"Couldn't fetch icon_url from {url}")
	# 			return None

	# 		# --- art id checks ---
	# 		t_art_url = t_schema.get('image').replace("_10.jpg", "_3")
	# 		async with self.lock:
	# 			# Check for cached urls
	# 			if t_art_url in self.seen_art_url:
	# 				return None
	# 			self.seen_art_url.add(t_art_url)

	# 		img = await self.s.get(t_art_url)
	# 		img_hash = hashlib.sha256(img.content).hexdigest()

	# 		async with self.lock:
	# 			# Check for hash dedupes
	# 			if img_hash in self.seen_hash:
	# 				return None
	# 			self.seen_hash.add(img_hash)

	# 		t_artist = nozero(t_schema.get('byArtist', {}).get('name'))
	# 		t_art_id = t_art_url.replace("https://f4.bcbits.com/img/a","").split("_", 1)[0]

	# 		return t_artist, t_art_id

### Metadata of one album

In [24]:
import json
album_schema = json.loads(soups[0].select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soups[0].select_one("[data-tralbum]")
tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}"))

[nozero(album_schema['name']), # album name
 nozero(album_schema['byArtist']['name']), # artist name
 album_schema['numTracks'], # number of tracks
 album_schema['keywords'] # tags
]

['Perli tal-Passat',
 'desert sand feels warm at night & days of blue skies',
 14,
 ['Electronic',
  'ambient',
  'chill',
  'experimental electronic',
  'memories',
  'rain',
  'slushwave',
  'smooth',
  'vaporwave',
  'United Kingdom']]

In [25]:
album_schema.keys()

dict_keys(['albumReleaseType', '@id', 'mainEntityOfPage', '@type', 'name', 'dateModified', 'albumRelease', 'byArtist', 'publisher', 'numTracks', 'track', 'image', 'keywords', 'datePublished', 'description', 'creditText', 'copyrightNotice', 'comment', 'sponsor', 'additionalProperty', '@context'])

In [26]:
tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'featured_track_id', 'initial_track_num', 'is_preorder', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'url', 'use_expando_lyrics'])

In [27]:
# Get all track urls
import isodate
import pandas as pd
df = pd.DataFrame([
	{
		"url": t['item']['@id']
	}
	for t in album_schema["track"]["itemListElement"]
])
print(df)

                                                  url
0        https://desertsand.bandcamp.com/track/mintix
1   https://desertsand.bandcamp.com/track/mog-dija...
2     https://desertsand.bandcamp.com/track/frammenti
3   https://desertsand.bandcamp.com/track/jiel-tal...
4   https://desertsand.bandcamp.com/track/modulazz...
5      https://desertsand.bandcamp.com/track/tfakkira
6   https://desertsand.bandcamp.com/track/meta-kie...
7    https://desertsand.bandcamp.com/track/solipsi-mu
8   https://desertsand.bandcamp.com/track/qatt-ma-...
9   https://desertsand.bandcamp.com/track/g-ajnejn...
10  https://desertsand.bandcamp.com/track/xorta-mi...
11  https://desertsand.bandcamp.com/track/kisi-tal...
12  https://desertsand.bandcamp.com/track/effu-jon...
13  https://desertsand.bandcamp.com/track/ritmu-ta...


#### Get unique track arts
This is how bandcamp art ID works:
1. Bandcamp generates a unique art_ID for each time you upload a track art.
2. The rest uses art_ID from the album art..

We can drop duplicates based on image hash, with some cases we have to delete manually (dobs album).

In [28]:
import re
import hashlib

async def get_art_id(url):
	session = BrowserSession()
	r = await session.get(url)
	soup = BeautifulSoup(r.text or "", "lxml")

	icon_url = soup.select_one('link[rel="shortcut icon"]')['href']

	img = await session.get(icon_url)
	img_hash = hashlib.sha256(img.content).hexdigest()

	art_id = re.search(r'a(\d+)_', icon_url).group(1)

	return art_id, img_hash


results = await asyncio.gather(
    *(get_art_id(url) for url in df["url"])
)

art_df = pd.DataFrame(
    results,
    columns=["art_id", "img_hash"]
)

print(art_df)

        art_id                                           img_hash
0   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
1   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
2   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
3   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
4   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
5   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
6   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
7   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
8   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
9   3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
10  3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
11  3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
12  3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...
13  3561606714  8200576d3dc0bbf34463c68e8308d88d976bea211646e3...


In [29]:
print(f"Total unique art IDs: {art_df['art_id'].nunique()}, Unique Image Hashes: {art_df['img_hash'].nunique()}")

Total unique art IDs: 1, Unique Image Hashes: 1


#### Get all dates of the album

In [30]:
tralbum = json.loads(tralbum_tag["data-tralbum"])['current']
tralbum2 = json.loads(tralbum_tag["data-tralbum"])['trackinfo']

In [31]:
[nozero(tralbum['title']), # album name
 tralbum['art_id'], # album image id: 2569838638 in https://f4.bcbits.com/img/a2569838638_10.jpg
 tralbum['new_date'], # date first created draft
 tralbum['mod_date'], # date last modified
 tralbum['publish_date'], # publication date
 tralbum['release_date'] # release date
 ]

['Perli tal-Passat',
 3561606714,
 '16 Dec 2024 00:19:06 GMT',
 '15 Jul 2025 14:55:17 GMT',
 '16 Dec 2024 17:49:51 GMT',
 '16 Dec 2024 00:00:00 GMT']

In [32]:
# Get all track durations
import pandas as pd
track_df = pd.DataFrame([
	{
		"position": t['track_num'],
		"duration": t['duration'],
	}
	for t in tralbum2
])
print(track_df)

    position  duration
0          1   543.115
1          2   751.118
2          3   600.377
3          4   435.155
4          5   644.972
5          6   636.311
6          7   549.659
7          8   645.083
8          9   726.842
9         10   631.239
10        11   535.385
11        12   438.447
12        13   818.684
13        14   880.626


#### Adding up total runtime of the album:

In [33]:
from datetime import timedelta
print(timedelta(seconds=int(track_df["duration"].sum())))

2:27:17
